# Camera simulator playground

Follow mode in MuJoCo with the simulated MaixCAM2 camera, and the JEPA encoders on its frames. See [CAMSIM.md](CAMSIM.md).

In the Docker lab (`docker compose up -d lab`, see `docker/README.md`) everything is installed and several people can edit this notebook at the same time. Outputs go to `/work/camsim`. Elsewhere, set `ODP_DIR` and `STOCK_ONNX` as in CAMSIM.md.

In [ ]:
import os, sys

from odm_train.camsim import gl
print("renderer:", gl.setup())  # before mujoco is imported

ODP_DIR = os.environ.get("ODP_DIR", "Open_Duck_Playground")
POLICY = os.environ.get("STOCK_ONNX", "Open_Duck_Mini/BEST_WALK_ONNX_2.onnx")  # or your own policy .onnx
OUT = "/work/camsim" if os.path.isdir("/work") else os.path.abspath("camsim_out")
os.makedirs(OUT, exist_ok=True)
os.chdir(ODP_DIR)  # the duck's XML and reference motion use relative paths
sys.path.insert(0, ODP_DIR)

import numpy as np
import matplotlib.pyplot as plt
from odm_train.camsim.follow_sim import FollowSim, Recorder, VideoWriter, draw_box
from odm_train.camsim.scene import CameraConfig, SceneConfig

## 1. Set up the camera and the scene

Change anything here and re-run: mount, field of view, frame rate, latency, distortion, sensor noise.

In [ ]:
cam = CameraConfig(mount="head", width=320, height=240, hfov_deg=70, hz=15, latency_ms=66, k1=-0.08)
scene = SceneConfig(distractor=True, seed=0)

if "sim" in globals():
    sim.close()
sim = FollowSim(POLICY, cam=cam, scene=scene, seed=0)

## 2. Step and look through the camera

In [ ]:
frames = []
for k in range(150):  # 3 s at 50 Hz
    info = sim.step()
    pkt = info["delivered"]
    if pkt is not None and k % 15 == 0:
        frames.append(draw_box(pkt["frame"].copy(), pkt["box"]))
    if sim.fallen:
        print("fell"); break

dist, bearing = sim.truth()
print(f"t={sim.t:.1f}s  person {dist:.2f} m away, {np.degrees(bearing):+.0f} deg  cmd={np.round(info['cmd'], 2)}")
fig, axes = plt.subplots(1, len(frames), figsize=(3 * len(frames), 2.4))
for ax, f in zip(np.atleast_1d(axes), frames):
    ax.imshow(f); ax.axis("off")

## 3. A whole episode as a video (head camera beside a third-person view)

In [ ]:
import mediapy

sim.reset()
video = VideoWriter(sim, os.path.join(OUT, "notebook_follow.mp4"), cam.hz)
for _ in range(int(10 / 0.02)):
    video.add(sim.step())
    if sim.fallen:
        break
video.close()
print(video.path)
mediapy.show_video(mediapy.read_video(video.path), fps=cam.hz)

## 4. Record frames, encode them, fit a projection

`standin_small` needs no download. Switch to `vjepa2_1_base` or `dinov2_small` for real features (downloaded once into `/work/.cache`).

In [ ]:
from odm_train.camsim.encoders import EmbeddingPipeline, Projection, make_encoder

ENCODER = "standin_small"   # "vjepa2_1_base", "dinov2_small", "onnx:/work/my_encoder.onnx"
enc = make_encoder(ENCODER)

rec = Recorder()
for ep in range(2):
    sim.reset()
    for _ in range(int(10 / 0.02)):
        rec.add(ep, sim.step())
data_path = os.path.join(OUT, "notebook_follow.npz")
print("frames:", rec.save(data_path))

z = np.load(data_path)
feats = enc(z["frame"])
proj = Projection.fit(feats, dim=16)
proj.save(os.path.join(OUT, "notebook_projection.npz"))
emb = proj(feats)
print(f"{enc.name}: {feats.shape[1]} features -> {emb.shape[1]} values, {100 * proj.explained.sum():.0f}% of variance")

Does the embedding know where the person is? Colour the first two values by the true bearing and distance.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, key, label in ((ax[0], "bearing", "bearing (deg)"), (ax[1], "dist", "distance (m)")):
    c = np.degrees(z[key]) if key == "bearing" else z[key]
    s = a.scatter(emb[:, 0], emb[:, 1], c=c, s=8, cmap="viridis")
    fig.colorbar(s, ax=a, label=label); a.set_xlabel("emb_0"); a.set_ylabel("emb_1")

## 5. Drive the sim with the encoder

The embedding fills the policy inputs `emb_<i>` when the policy's profile lists them.

In [ ]:
sim.close()
sim = FollowSim(POLICY, cam=cam, scene=scene, seed=1, encoder=EmbeddingPipeline(enc, proj))
for _ in range(100):
    info = sim.step()
    if info["delivered"] is not None:
        last = info["delivered"]["emb"]
print("latest embedding:", np.round(last[:6], 3), "...")